# 🏆 IVF 임신 성공 예측 — 8-Model Ensemble 완전 재현

**DACON 해커톤 3등 솔루션 · OOF 0.74101 · Public LB 0.74235**

---

## 이 노트북의 위치

원본 프로젝트는 `src/` 폴더의 11개 `.py` 파일로 구성됩니다.
이 노트북은 그 파일들을 **실행 순서대로 단일 ipynb에 통합**한 것

## 통합된 원본 파일

| # | 파일 | 역할 | 노트북 섹션 |
|---|---|---|---|
| 1 | `src/features/v2_features.py` | 142 파생변수 | §3 |
| 2 | `src/features/oof_target_encoder.py` | Leakage-safe TE | §4 |
| 3 | `src/train_v1_optunaCAT.py` | v1 학습 (default LGBM/XGB + Optuna CAT) | §6 |
| 4 | `src/train_v2_full.py` | v2 학습 (Optuna LGBM/XGB/CAT) | §7 |
| 5 | `src/train_mlp.py` | MLP 5-fold | §8 |
| 6 | `src/train_tabnet.py` | TabNet 5-fold | §9 |
| 7 | `src/blend_8model_tabnet.py` | Nelder-Mead 8-blend | §10 |
| 8 | `src/tune_optuna.py` | CatBoost Optuna (참고) | Appendix A |
| 9 | `src/tune_optuna_lgbm.py` | LightGBM Optuna (참고) | Appendix A |
| 10 | `src/tune_optuna_xgb.py` | XGBoost Optuna (참고) | Appendix A |
| 11 | `src/train_ft_transformer.py` | FT-Transformer (실험) | Appendix B |

> **부록(Appendix)은 선택 실행입니다.** Optuna 결과는 이미 hardcoded되어 있으므로 §6-7에서 그대로 사용됩니다. FT-Transformer는 8-blend 구성에 포함되지 않습니다.

## 디렉토리 준비

```
project_root/
├── data/
│   ├── train.csv
│   ├── test.csv
│   └── sample_submission.csv
├── models/             ← 자동 생성, oof/test npz 저장됨
├── submission/         ← 자동 생성, submission_8blend_tabnet.csv 출력
└── notebook.ipynb      ← 이 파일
```

## 실행 시간 (Mac M2 Pro 기준 PDF p.20)

| 단계 | 시간 |
|---|---|
| §1-5 정의 + FE | ~1 분 |
| §6 v1 학습 (3 GBM × 5-fold) | ~25-35 분 |
| §7 v2 학습 (3 GBM × 5-fold) | ~25-35 분 |
| §8 MLP 5-fold | ~8-10 분 |
| §9 TabNet 5-fold | ~30-50 분 |
| §10 8-blend | < 10 초 |
| **전체** | **약 90-120 분** |

빠르게 검증만 하고 싶다면 §6, §7의 `--catboost-only` 옵션 사용 가능.


## 1. 환경 설정 (imports, seed, 경로)

In [1]:
# 최초 1회만 실행 (필요시 주석 해제)
# !pip install lightgbm xgboost catboost scikit-learn pandas numpy scipy joblib
# !pip install torch torchvision      # PyTorch (Mac M2는 별도 설치 가이드 참고)
# !pip install pytorch-tabnet
# !pip install optuna                  # Appendix A 만 사용 시


In [2]:
from __future__ import annotations
import argparse, gc, json, time, warnings, sys
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib
from scipy.optimize import minimize
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool

SEED = 42
np.random.seed(SEED)

# 경로 자동 탐색 (notebook 위치 기준)
ROOT = Path.cwd()
for _ in range(3):
    if (ROOT / "data" / "train.csv").exists():
        break
    ROOT = ROOT.parent
else:
    raise FileNotFoundError(
        "data/train.csv 가 없습니다. 노트북을 프로젝트 루트(또는 그 하위)에 두세요."
    )

DATA = ROOT / "data"
MODELS = ROOT / "models"
SUB = ROOT / "submission"
MODELS.mkdir(exist_ok=True)
SUB.mkdir(exist_ok=True)

TARGET = "임신 성공 여부"
ID_COL = "ID"
N_SPLITS = 5
TARGET_ENC_COLS = ["시술 시기 코드", "특정 시술 유형", "배란 유도 유형", "배아 생성 주요 이유"]

print(f"✅ ROOT     : {ROOT}")
print(f"✅ DATA     : {DATA}")
print(f"✅ MODELS   : {MODELS}")
print(f"✅ SUB      : {SUB}")


✅ ROOT     : /Users/admin/project_4
✅ DATA     : /Users/admin/project_4/data
✅ MODELS   : /Users/admin/project_4/models
✅ SUB      : /Users/admin/project_4/submission


## 2. 데이터 로딩 & sanity check

In [3]:
train_raw = pd.read_csv(DATA / "train.csv")
test_raw  = pd.read_csv(DATA / "test.csv")
sub_raw   = pd.read_csv(DATA / "sample_submission.csv")

print(f"train_raw : {train_raw.shape}")
print(f"test_raw  : {test_raw.shape}")
print(f"sub_raw   : {sub_raw.shape}")

assert TARGET in train_raw.columns
assert TARGET not in test_raw.columns
assert len(set(train_raw[ID_COL]) & set(test_raw[ID_COL])) == 0

print(f"\n임신 성공률 : {train_raw[TARGET].mean():.4%}")


train_raw : (256351, 69)
test_raw  : (90067, 68)
sub_raw   : (90067, 2)

임신 성공률 : 25.8349%


## 3. v2 Feature Engineering — 7 카테고리 142 파생변수

`src/features/v2_features.py` 원본 코드 전체.

In [4]:
"""
v2_features.py
=====================================================================
8 함수, ~80개 신규 피처. 모두 row-wise 변환으로 leakage 위험 0.

함수 구성:
  add_age_features          — §1 나이 파생 + 중앙값 매핑
  add_history_features       — §2 횟수·이력 ratio
  add_treatment_token_features — §3 ICSI/IVF/IUI/BLASTOCYST/AH/FER 토큰
  add_embryo_efficiency      — §4 배아·난자 효율 ratio + log1p
  add_transfer_strategy      — §5 transfer_day_bin, fresh × frozen pattern
  add_cause_aggregates_v2    — §6 남/여/부부 cause 집계
  add_donor_ordinals         — §7 난자/정자 기증자 나이 ordinal
  add_missing_signals        — §8 informative NaN flags

통합 함수: add_v2_all_features(df) — 위 8개 일괄 적용
"""

# ========================================================================
# 매핑
# ========================================================================
AGE_TO_ORDINAL = {
    "만18-34세": 0, "만35-37세": 1, "만38-39세": 2,
    "만40-42세": 3, "만43-44세": 4, "만45-50세": 5, "알 수 없음": -1,
}
AGE_TO_MID = {
    "만18-34세": 26, "만35-37세": 36, "만38-39세": 38.5,
    "만40-42세": 41, "만43-44세": 43.5, "만45-50세": 47.5, "알 수 없음": np.nan,
}
COUNT_TO_INT = {"0회":0, "1회":1, "2회":2, "3회":3, "4회":4, "5회":5, "6회 이상":6}

DONOR_OOCYTE_AGE_MAP = {"만20세 이하":0, "만21-25세":1, "만26-30세":2, "만31-35세":3, "알 수 없음":-1}
DONOR_SPERM_AGE_MAP = {"만20세 이하":0, "만21-25세":1, "만26-30세":2, "만31-35세":3,
                       "만36-40세":4, "만41-45세":5, "알 수 없음":-1}

COUNT_COLS = ["총 시술 횟수", "클리닉 내 총 시술 횟수", "IVF 시술 횟수", "DI 시술 횟수",
              "총 임신 횟수", "IVF 임신 횟수", "DI 임신 횟수",
              "총 출산 횟수", "IVF 출산 횟수", "DI 출산 횟수"]

# 토큰 (§3)
TREATMENT_TOKENS = ["ICSI", "IVF", "IUI", "BLASTOCYST", "AH", "FER", "Unknown"]

# Informative NaN columns (§5/§8 EDA 검증)
INFORMATIVE_NA_COLS = [
    "배아 이식 경과일",            # Δ=-28.3%p (cancelled cycle)
    "난자 혼합 경과일",            # Δ=-8.8%p
    "임신 시도 또는 마지막 임신 경과 연수",  # Δ=+4.3%p
    "착상 전 유전 검사 사용 여부",
    "배아 해동 경과일",
    "난자 채취 경과일",
]


# ========================================================================
# §1. Age features
# ========================================================================
def add_age_features(df: pd.DataFrame) -> pd.DataFrame:
    """나이 파생: ordinal, mid-age, threshold flags, unknown."""
    df = df.copy()
    age = df.get("시술 당시 나이", pd.Series("알 수 없음", index=df.index))
    df["age_ord"] = age.map(AGE_TO_ORDINAL).fillna(-1).astype(int)
    df["age_mid"] = age.map(AGE_TO_MID).astype(float)
    df["age_unknown"] = (age == "알 수 없음").astype(int)
    df["age_35p"] = (df["age_ord"] >= 1).astype(int)
    df["age_38p"] = (df["age_ord"] >= 2).astype(int)
    df["age_40p"] = (df["age_ord"] >= 3).astype(int)
    df["age_43p"] = (df["age_ord"] >= 4).astype(int)
    return df


# ========================================================================
# §2. History features
# ========================================================================
def add_history_features(df: pd.DataFrame) -> pd.DataFrame:
    """횟수 ordinal + ratio + prior flags."""
    df = df.copy()
    # 횟수 → 정수
    for c in COUNT_COLS:
        if c in df.columns:
            df[f"{c}_int"] = df[c].map(COUNT_TO_INT).fillna(-1).astype(int)
            df[f"{c}_is_censored"] = (df[c] == "6회 이상").astype(int)

    # Prior flags
    if "총 임신 횟수_int" in df.columns:
        df["prior_pregnancy_any"] = (df["총 임신 횟수_int"] >= 1).astype(int)
    if "총 출산 횟수_int" in df.columns:
        df["prior_live_birth_any"] = (df["총 출산 횟수_int"] >= 1).astype(int)
    if "총 임신 횟수_int" in df.columns and "총 출산 횟수_int" in df.columns:
        df["prior_pregnancy_no_live_birth"] = (
            (df["총 임신 횟수_int"] >= 1) & (df["총 출산 횟수_int"] == 0)
        ).astype(int)

    # Ratios (with NaN for divide-by-zero)
    def safe_ratio(num_col, den_col, name):
        if num_col in df.columns and den_col in df.columns:
            den = df[den_col].replace(0, np.nan)
            df[name] = (df[num_col] / den).fillna(-1).clip(-1, 5)

    safe_ratio("총 임신 횟수_int", "총 시술 횟수_int", "pregnancy_per_treatment")
    safe_ratio("총 출산 횟수_int", "총 시술 횟수_int", "live_birth_per_treatment")
    safe_ratio("총 출산 횟수_int", "총 임신 횟수_int", "live_birth_per_pregnancy")
    safe_ratio("IVF 임신 횟수_int", "IVF 시술 횟수_int", "ivf_preg_per_ivf_tr")
    safe_ratio("DI 임신 횟수_int", "DI 시술 횟수_int", "di_preg_per_di_tr")
    safe_ratio("IVF 출산 횟수_int", "IVF 시술 횟수_int", "ivf_birth_per_ivf_tr")

    return df


# ========================================================================
# §3. Treatment token features
# ========================================================================
def add_treatment_token_features(df: pd.DataFrame) -> pd.DataFrame:
    """특정 시술 유형에서 ICSI/IVF/IUI/BLASTOCYST/AH/FER/Unknown 추출."""
    df = df.copy()
    if "특정 시술 유형" not in df.columns:
        return df
    s = df["특정 시술 유형"].fillna("").astype(str).str.upper()
    for tok in TREATMENT_TOKENS:
        df[f"treat_has_{tok.lower()}"] = s.str.contains(tok.upper(), regex=False).astype(int)
    df["treat_token_count"] = sum(
        df[f"treat_has_{tok.lower()}"] for tok in TREATMENT_TOKENS
    )
    # 첫 토큰 추출 (단순 categorical)
    df["treat_main_first"] = df["특정 시술 유형"].fillna("UNKNOWN").astype(str).str.split('/').str[0]
    return df


# ========================================================================
# §4. Embryo efficiency
# ========================================================================
NUM_EMB_COLS = [
    "총 생성 배아 수", "미세주입된 난자 수", "미세주입에서 생성된 배아 수",
    "이식된 배아 수", "미세주입 배아 이식 수", "저장된 배아 수", "미세주입 후 저장된 배아 수",
    "해동된 배아 수", "해동 난자 수", "수집된 신선 난자 수", "저장된 신선 난자 수",
    "혼합된 난자 수", "파트너 정자와 혼합된 난자 수", "기증자 정자와 혼합된 난자 수",
]


def add_embryo_efficiency(df: pd.DataFrame) -> pd.DataFrame:
    """배아·난자 수치 log1p + zero/missing flags + 효율 ratio."""
    df = df.copy()

    # log1p + flags
    for c in NUM_EMB_COLS:
        if c in df.columns:
            v = df[c].fillna(0).clip(lower=0)
            df[f"{c}_log1p"] = np.log1p(v)
            df[f"{c}_is_zero"] = (df[c] == 0).astype(int)
            df[f"{c}_is_missing"] = df[c].isna().astype(int)

    # Ratios — divide-by-zero → -1 sentinel
    def ratio(num, den, name):
        if num in df.columns and den in df.columns:
            df[name] = np.where(
                df[den].fillna(0) > 0,
                df[num].fillna(0) / df[den].clip(lower=1e-9),
                -1.0,
            )

    ratio("이식된 배아 수", "총 생성 배아 수", "ratio_transferred_per_created")
    ratio("저장된 배아 수", "총 생성 배아 수", "ratio_stored_per_created")
    ratio("총 생성 배아 수", "혼합된 난자 수", "ratio_created_per_mixed")
    ratio("미세주입에서 생성된 배아 수", "미세주입된 난자 수", "ratio_icsi_efficiency")
    ratio("파트너 정자와 혼합된 난자 수", "혼합된 난자 수", "ratio_partner_sperm")
    ratio("기증자 정자와 혼합된 난자 수", "혼합된 난자 수", "ratio_donor_sperm")

    # Surplus / shortage
    if "총 생성 배아 수" in df.columns and "이식된 배아 수" in df.columns:
        df["embryos_surplus"] = (
            df["총 생성 배아 수"].fillna(0) - df["이식된 배아 수"].fillna(0)
        ).clip(lower=0)

    if "총 생성 배아 수" in df.columns and "저장된 배아 수" in df.columns and "이식된 배아 수" in df.columns:
        df["embryos_unused"] = (
            df["총 생성 배아 수"].fillna(0)
            - df["이식된 배아 수"].fillna(0)
            - df["저장된 배아 수"].fillna(0)
        ).clip(lower=0)

    return df


# ========================================================================
# §5. Transfer strategy
# ========================================================================
def add_transfer_strategy(df: pd.DataFrame) -> pd.DataFrame:
    """transfer_day_bin, fresh × frozen pattern, blastocyst × day."""
    df = df.copy()

    # has_transfer
    if "배아 이식 경과일" in df.columns:
        df["has_embryo_transfer"] = df["배아 이식 경과일"].notna().astype(int)
        df["cancelled_cycle"] = df["배아 이식 경과일"].isna().astype(int)

        # Day bins
        d = df["배아 이식 경과일"]
        df["transfer_day0_2"] = ((d >= 0) & (d <= 2)).astype(int)
        df["transfer_day3"] = (d == 3).astype(int)
        df["transfer_day5"] = (d == 5).astype(int)
        df["transfer_day_blastocyst"] = ((d >= 5) & (d <= 6)).astype(int)

    # Embryo transfer count bins
    if "이식된 배아 수" in df.columns:
        n = df["이식된 배아 수"].fillna(-1)
        df["transfer_count_0"] = (n == 0).astype(int)
        df["transfer_count_1"] = (n == 1).astype(int)
        df["transfer_count_2"] = (n == 2).astype(int)
        df["transfer_count_3p"] = (n >= 3).astype(int)
        df["transfer_count_NA"] = (n == -1).astype(int)

    # Fresh × Frozen × Donor pattern
    if all(c in df.columns for c in ["신선 배아 사용 여부", "동결 배아 사용 여부", "기증 배아 사용 여부"]):
        f = df["신선 배아 사용 여부"].fillna(-1).astype(str)
        fz = df["동결 배아 사용 여부"].fillna(-1).astype(str)
        dn = df["기증 배아 사용 여부"].fillna(-1).astype(str)
        df["fresh_frozen_donor_pattern"] = "F" + f + "_Z" + fz + "_D" + dn

    # Blastocyst × day5 (cleavage vs blastocyst proxy via day)
    if "transfer_day5" in df.columns and "treat_has_blastocyst" in df.columns:
        df["blastocyst_x_day5"] = df["transfer_day5"] * df["treat_has_blastocyst"]

    return df


# ========================================================================
# §6. Cause aggregates v2
# ========================================================================
def add_cause_aggregates_v2(df: pd.DataFrame) -> pd.DataFrame:
    """남/여/부부/sperm cause counts + binary aggregates."""
    df = df.copy()
    cause_cols = [c for c in df.columns if "불임 원인" in c]
    binary_cause = [c for c in cause_cols if df[c].dropna().isin([0,1]).all()]

    if not binary_cause:
        return df

    df["cause_total_count"] = df[binary_cause].sum(axis=1)
    df["cause_any"] = (df["cause_total_count"] > 0).astype(int)
    df["cause_none"] = (df["cause_total_count"] == 0).astype(int)
    df["cause_multi"] = (df["cause_total_count"] >= 2).astype(int)

    male_cols = [c for c in binary_cause if any(k in c for k in ["남성","정자"])]
    female_cols = [c for c in binary_cause if any(k in c for k in ["여성","난관","자궁","배란"])]
    couple_cols = [c for c in binary_cause if "부부" in c]
    sperm_cols = [c for c in binary_cause if "정자" in c]

    df["cause_male_count"] = df[male_cols].sum(axis=1) if male_cols else 0
    df["cause_female_count"] = df[female_cols].sum(axis=1) if female_cols else 0
    df["cause_couple_count"] = df[couple_cols].sum(axis=1) if couple_cols else 0
    df["cause_sperm_count"] = df[sperm_cols].sum(axis=1) if sperm_cols else 0

    df["cause_male_any"] = (df["cause_male_count"] > 0).astype(int)
    df["cause_female_any"] = (df["cause_female_count"] > 0).astype(int)
    df["cause_male_female_both"] = (
        (df["cause_male_any"] == 1) & (df["cause_female_any"] == 1)
    ).astype(int)

    if "불명확 불임 원인" in df.columns:
        df["unexplained_or_none"] = (
            (df["cause_total_count"] == 0) | (df["불명확 불임 원인"] == 1)
        ).astype(int)

    return df


# ========================================================================
# §7. Donor ordinals
# ========================================================================
def add_donor_ordinals(df: pd.DataFrame) -> pd.DataFrame:
    """난자/정자 기증자 나이 ordinal + cross with recipient age."""
    df = df.copy()

    if "난자 기증자 나이" in df.columns:
        df["egg_donor_age_ord"] = df["난자 기증자 나이"].map(DONOR_OOCYTE_AGE_MAP).fillna(-1).astype(int)
        df["egg_donor_age_unknown"] = (df["난자 기증자 나이"] == "알 수 없음").astype(int)
        df["egg_donor_age_optimal"] = df["난자 기증자 나이"].isin(["만21-25세", "만26-30세"]).astype(int)

    if "정자 기증자 나이" in df.columns:
        df["sperm_donor_age_ord"] = df["정자 기증자 나이"].map(DONOR_SPERM_AGE_MAP).fillna(-1).astype(int)
        df["sperm_donor_age_unknown"] = (df["정자 기증자 나이"] == "알 수 없음").astype(int)

    # Source flags (egg, sperm)
    if "난자 출처" in df.columns:
        df["uses_donor_oocyte"] = (df["난자 출처"] == "기증 제공").astype(int)
    if "정자 출처" in df.columns:
        df["uses_donor_sperm"] = (df["정자 출처"] == "기증 제공").astype(int)

    # Cross: 난자 출처 × 정자 출처
    if "난자 출처" in df.columns and "정자 출처" in df.columns:
        df["egg_sperm_source_cross"] = (
            df["난자 출처"].fillna("NA").astype(str) + "_" +
            df["정자 출처"].fillna("NA").astype(str)
        )

    # age × egg source interaction
    if "age_ord" in df.columns and "uses_donor_oocyte" in df.columns:
        df["age_x_egg_donor"] = df["age_ord"] * df["uses_donor_oocyte"]
        df["age_x_no_egg_donor"] = df["age_ord"] * (1 - df["uses_donor_oocyte"])

    return df


# ========================================================================
# §8. Missing signals
# ========================================================================
def add_missing_signals(df: pd.DataFrame) -> pd.DataFrame:
    """Informative NaN flags + row-level missing count."""
    df = df.copy()
    flags = []
    for col in INFORMATIVE_NA_COLS:
        if col in df.columns:
            flag = f"{col}_isna"
            df[flag] = df[col].isna().astype(int)
            flags.append(flag)
    if flags:
        df["sum_informative_na"] = df[flags].sum(axis=1)

    # 행 단위 missing count
    df["row_missing_count"] = df.isna().sum(axis=1)

    return df


# ========================================================================
# §9. Selected interactions
# ========================================================================
def add_priority_interactions(df: pd.DataFrame) -> pd.DataFrame:
    """우선순위 interaction 피처 (§9 prompt)."""
    df = df.copy()

    age = df.get("age_ord", pd.Series(0, index=df.index))
    n_transfer = df.get("이식된 배아 수", pd.Series(0, index=df.index)).fillna(0)
    transfer_day = df.get("배아 이식 경과일", pd.Series(0, index=df.index)).fillna(0)

    df["age_x_n_transfer"] = age * n_transfer
    df["age_x_transfer_day"] = age * transfer_day

    if "treat_has_icsi" in df.columns:
        df["age_x_icsi"] = age * df["treat_has_icsi"]
    if "treat_has_blastocyst" in df.columns:
        df["age_x_blastocyst"] = age * df["treat_has_blastocyst"]
    if "동결 배아 사용 여부" in df.columns:
        df["age_x_FET"] = age * df["동결 배아 사용 여부"].fillna(0).astype(int)

    return df


# ========================================================================
# 통합
# ========================================================================
def add_v2_all_features(df: pd.DataFrame) -> pd.DataFrame:
    """모든 v2 피처를 일괄 적용 (순서 중요: age → history → token → embryo → ...)."""
    df = add_age_features(df)
    df = add_history_features(df)
    df = add_treatment_token_features(df)
    df = add_embryo_efficiency(df)
    df = add_transfer_strategy(df)
    df = add_cause_aggregates_v2(df)
    df = add_donor_ordinals(df)
    df = add_missing_signals(df)
    df = add_priority_interactions(df)
    return df


# ========================================================================
# Smoke test
# ========================================================================

print("✅ v2 feature functions defined (8 + priority interactions)")

✅ v2 feature functions defined (8 + priority interactions)


## 4. OOFTargetEncoder — Leakage-safe Target Encoding

`src/features/oof_target_encoder.py` 원본 코드 전체.

- `fit_transform_oof()` — fold 내부 train으로만 fit → train/val transform
- `fit_full_then_transform()` — 전체 train labels로 fit → test transform (test labels 미사용)

In [5]:
"""
oof_target_encoder.py — Leakage-safe target encoding
========================================================
§10 leakage audit의 모든 항목 통과:
  - target encoding은 fold 내부 train으로만 fit
  - test set 인코딩은 전체 train labels 사용 (test labels 사용 안함)
  - smoothing으로 rare category 안정화
  - test에 없는 category는 global_mean으로 채움

사용법:
    enc = OOFTargetEncoder(cols=['시술 시기 코드','특정 시술 유형'], smoothing=50)
    # Fold-level encoding (during CV)
    X_tr_te, X_va_te = enc.fit_transform_oof(X_tr, y_tr, X_va)

    # Test encoding (after final model)
    X_test_te = enc.fit_full_then_transform(X_train_full, y_full, X_test)
"""


class OOFTargetEncoder:
    def __init__(self, cols: list, smoothing: float = 50.0):
        self.cols = cols
        self.smoothing = smoothing

    def _encode(self, train_df: pd.DataFrame, y_train: pd.Series, target_df: pd.DataFrame):
        """train_df + y_train에서 평균 학습, target_df에 적용."""
        global_mean = float(y_train.mean())
        out = pd.DataFrame(index=target_df.index)

        for col in self.cols:
            if col not in train_df.columns:
                continue
            s_tr = train_df[col].astype(str)
            s_te = target_df[col].astype(str)

            agg = (
                pd.DataFrame({"y": y_train.values, "g": s_tr.values})
                .groupby("g")["y"]
                .agg(["mean", "count"])
            )
            smoothed = (
                agg["mean"] * agg["count"] + global_mean * self.smoothing
            ) / (agg["count"] + self.smoothing)

            out[f"{col}_te"] = s_te.map(smoothed).fillna(global_mean).astype(float)

        return out

    def fit_transform_oof(
        self,
        X_tr: pd.DataFrame,
        y_tr: pd.Series,
        X_va: pd.DataFrame,
    ) -> tuple[pd.DataFrame, pd.DataFrame]:
        """fold 내부 train으로 fit → train/val 모두 transform."""
        train_te = self._encode(X_tr, y_tr, X_tr)
        val_te = self._encode(X_tr, y_tr, X_va)
        return train_te, val_te

    def fit_full_then_transform(
        self,
        X_full: pd.DataFrame,
        y_full: pd.Series,
        X_test: pd.DataFrame,
    ) -> pd.DataFrame:
        """전체 train으로 fit → test transform (test labels 사용 안함)."""
        return self._encode(X_full, y_full, X_test)


print("✅ OOFTargetEncoder defined")

✅ OOFTargetEncoder defined


## 5. Feature Engineering 적용 & 공통 유틸

v2 features 적용 → train/test 컬럼 정렬 → XGBoost용 정수-코드 universe 사전 구축 → test set TE.

이후 §6-9의 모든 모델이 이 데이터를 사용합니다.

In [6]:
# ── train_v2_full.py 의 PATCH 헬퍼 함수들 (v1, v2 모두 사용) ──
def _is_object_or_string(s):
    return s.dtype == "object" or pd.api.types.is_string_dtype(s)

def get_object_columns(df):
    return [c for c in df.columns if _is_object_or_string(df[c])]

def build_object_value_universe(X_full, X_test):
    universe = {}
    for c in get_object_columns(X_full):
        if c not in X_test.columns:
            continue
        train_vals = X_full[c].astype(str).fillna("__NA__").unique().tolist()
        test_vals  = X_test[c].astype(str).fillna("__NA__").unique().tolist()
        all_vals = sorted(set(train_vals) | set(test_vals))
        universe[c] = {v: i for i, v in enumerate(all_vals)}
    return universe

def encode_with_universe(X, universe):
    X = X.copy()
    for col, mapping in universe.items():
        if col in X.columns:
            X[col] = (X[col].astype(str).fillna("__NA__")
                      .map(mapping).fillna(-1).astype(np.int32))
    return X

def prep_for_lgbm(X):
    X = X.copy()
    for c in get_object_columns(X):
        X[c] = X[c].astype("category")
    return X

def prep_for_cat(X, cat_features):
    X = X.copy()
    for c in cat_features:
        if c in X.columns:
            X[c] = X[c].astype(str).fillna("missing")
    return X


In [7]:
# ── v2 features 적용 ──
print("v2 features 적용 중...")
t0 = time.time()
train = add_v2_all_features(train_raw)
test  = add_v2_all_features(test_raw)
print(f"  train: {train_raw.shape} → {train.shape}")
print(f"  test : {test_raw.shape} → {test.shape}")
print(f"  소요 : {time.time()-t0:.1f}s")

# ── target / id 분리 ──
y = train[TARGET].astype(int).reset_index(drop=True)
X      = train.drop(columns=[c for c in [TARGET, ID_COL] if c in train.columns]).reset_index(drop=True)
X_test = test.drop(columns=[ID_COL] if ID_COL in test.columns else []).reset_index(drop=True)

common = [c for c in X.columns if c in X_test.columns]
only_train = sorted(set(X.columns) - set(X_test.columns))
if only_train:
    print(f"⚠️  train-only 컬럼 제거: {only_train}")
X = X[common]
X_test = X_test[common]

# ── XGBoost용 정수-코드 universe ──
obj_universe = build_object_value_universe(X, X_test)
X_xgb = encode_with_universe(X, obj_universe)
X_test_xgb = encode_with_universe(X_test, obj_universe)
assert len(get_object_columns(X_xgb)) == 0, "X_xgb에 object 잔존"
print(f"\nObject cols mapped (XGB universe): {len(obj_universe)}")

# ── Test set TE (전체 train labels, test labels 미사용) ──
te_cols_present = [c for c in TARGET_ENC_COLS if c in X.columns]
enc_test = OOFTargetEncoder(cols=te_cols_present, smoothing=50.0)
X_test_te = enc_test.fit_full_then_transform(X, y, X_test)

X_test     = pd.concat([X_test.reset_index(drop=True),     X_test_te.reset_index(drop=True)], axis=1)
X_test_xgb = pd.concat([X_test_xgb.reset_index(drop=True), X_test_te.reset_index(drop=True)], axis=1)

print(f"\nX            : {X.shape}")
print(f"X_test       : {X_test.shape}")
print(f"X_xgb        : {X_xgb.shape}")
print(f"X_test_xgb   : {X_test_xgb.shape}")
print(f"TE columns   : {te_cols_present}")

del train_raw, test_raw, train, test
gc.collect()


v2 features 적용 중...
  train: (256351, 69) → (256351, 212)
  test : (90067, 68) → (90067, 211)
  소요 : 3.5s

Object cols mapped (XGB universe): 23

X            : (256351, 210)
X_test       : (90067, 214)
X_xgb        : (256351, 210)
X_test_xgb   : (90067, 214)
TE columns   : ['시술 시기 코드', '특정 시술 유형', '배란 유도 유형', '배아 생성 주요 이유']


16

## 5b. 공통 5-fold GBM 학습 함수

`train_v1_optunaCAT.py`와 `train_v2_full.py`는 fold 루프가 100% 동일하고
**LGBM_PARAMS·XGB_PARAMS만 다릅니다.** 따라서 함수로 추상화합니다.

In [8]:
def train_gbm_5fold(X, y, X_test, X_xgb, X_test_xgb,
                    lgbm_params, xgb_params, cat_params,
                    te_cols, n_splits=5, seed=42, label="v?",
                    save_models=False):
    """원본 train_v1_optunaCAT.py / train_v2_full.py 의 fold 루프 그대로."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    oof_lgbm = np.zeros(len(X), dtype=np.float32)
    oof_xgb  = np.zeros(len(X), dtype=np.float32)
    oof_cat  = np.zeros(len(X), dtype=np.float32)
    test_pred_lgbm = np.zeros(len(X_test),     dtype=np.float32)
    test_pred_xgb  = np.zeros(len(X_test_xgb), dtype=np.float32)
    test_pred_cat  = np.zeros(len(X_test),     dtype=np.float32)

    enc = OOFTargetEncoder(cols=te_cols, smoothing=50.0)

    for f_i, (tr, va) in enumerate(skf.split(X, y), start=1):
        t = time.time()
        Xtr_base = X.iloc[tr].reset_index(drop=True)
        Xva_base = X.iloc[va].reset_index(drop=True)
        Xtr_xgb_base = X_xgb.iloc[tr].reset_index(drop=True)
        Xva_xgb_base = X_xgb.iloc[va].reset_index(drop=True)
        ytr = y.iloc[tr].reset_index(drop=True)
        yva = y.iloc[va].reset_index(drop=True)

        # OOF target encoding (fold-internal train only)
        tr_te, va_te = enc.fit_transform_oof(Xtr_base, ytr, Xva_base)
        Xtr     = pd.concat([Xtr_base,     tr_te], axis=1)
        Xva     = pd.concat([Xva_base,     va_te], axis=1)
        Xtr_xgb = pd.concat([Xtr_xgb_base, tr_te], axis=1)
        Xva_xgb = pd.concat([Xva_xgb_base, va_te], axis=1)

        # ── LightGBM ──
        Xtr_l = prep_for_lgbm(Xtr); Xva_l = prep_for_lgbm(Xva)
        dtr = lgb.Dataset(Xtr_l, ytr)
        dva = lgb.Dataset(Xva_l, yva, reference=dtr)
        m = lgb.train(lgbm_params, dtr, num_boost_round=10000,
                      valid_sets=[dva],
                      callbacks=[lgb.early_stopping(200), lgb.log_evaluation(0)])
        oof_lgbm[va] = m.predict(Xva_l, num_iteration=m.best_iteration)
        X_test_l = prep_for_lgbm(X_test)
        test_pred_lgbm += m.predict(X_test_l, num_iteration=m.best_iteration).astype(np.float32) / n_splits
        if save_models:
            joblib.dump(m, MODELS / f"lgbm_{label}_fold{f_i}.pkl")
        del Xtr_l, Xva_l, X_test_l, m, dtr, dva; gc.collect()

        # ── XGBoost (정수 코드 입력) ──
        dtr = xgb.DMatrix(Xtr_xgb, label=ytr)
        dva = xgb.DMatrix(Xva_xgb, label=yva)
        m = xgb.train(xgb_params, dtr, num_boost_round=10000,
                      evals=[(dva, "valid")], early_stopping_rounds=200, verbose_eval=0)
        best_iter = m.best_iteration
        oof_xgb[va] = m.predict(dva, iteration_range=(0, best_iter + 1))
        dtest = xgb.DMatrix(X_test_xgb)
        test_pred_xgb += m.predict(dtest, iteration_range=(0, best_iter + 1)).astype(np.float32) / n_splits
        if save_models:
            m.save_model(str(MODELS / f"xgb_{label}_fold{f_i}.json"))
        del m, dtr, dva, dtest; gc.collect()

        # ── CatBoost (native object) ──
        cat_features = get_object_columns(Xtr)
        Xtr_c = prep_for_cat(Xtr, cat_features)
        Xva_c = prep_for_cat(Xva, cat_features)
        ptr = Pool(Xtr_c, ytr, cat_features=cat_features)
        pva = Pool(Xva_c, yva, cat_features=cat_features)
        m = CatBoostClassifier(**cat_params)
        m.fit(ptr, eval_set=pva, use_best_model=True, verbose=False)
        oof_cat[va] = m.predict_proba(pva)[:, 1]
        X_test_c = prep_for_cat(X_test, cat_features)
        ptest = Pool(X_test_c, cat_features=cat_features)
        test_pred_cat += m.predict_proba(ptest)[:, 1].astype(np.float32) / n_splits
        if save_models:
            m.save_model(str(MODELS / f"cat_{label}_fold{f_i}.cbm"))
        del Xtr_c, Xva_c, X_test_c, m, ptr, pva, ptest; gc.collect()

        a_l = roc_auc_score(yva, oof_lgbm[va])
        a_x = roc_auc_score(yva, oof_xgb[va])
        a_c = roc_auc_score(yva, oof_cat[va])
        print(f"  [{label}] Fold {f_i}: LGBM={a_l:.5f}  XGB={a_x:.5f}  CAT={a_c:.5f}  "
              f"({time.time()-t:.0f}s)", flush=True)

        del Xtr_base, Xva_base, Xtr_xgb_base, Xva_xgb_base
        del Xtr, Xva, Xtr_xgb, Xva_xgb
        gc.collect()

    print(f"  [{label}] OOF  LGBM={roc_auc_score(y, oof_lgbm):.5f}  "
          f"XGB={roc_auc_score(y, oof_xgb):.5f}  CAT={roc_auc_score(y, oof_cat):.5f}\n")
    return oof_lgbm, oof_xgb, oof_cat, test_pred_lgbm, test_pred_xgb, test_pred_cat


print("✅ train_gbm_5fold defined")


✅ train_gbm_5fold defined


## 6. v1 모델 학습 — `train_v1_optunaCAT.py`

원본 파라미터 그대로:
- LightGBM: `lr=0.03, num_leaves=127, ...` (가벼운 수동 튜닝)
- XGBoost: `lr=0.03, max_depth=7, ...` (가벼운 수동 튜닝)
- CatBoost: Optuna 결과 (`tune_optuna.py` 참조)

저장: `models/oof_v1_optunaCAT.npz`, `models/test_v1_optunaCAT.npz`

> **기대 OOF**: lgb ≈ 0.7394, xgb ≈ 0.7402, cat ≈ 0.7404 (PDF p.20)

In [9]:
# train_v1_optunaCAT.py 원본 파라미터
V1_LGBM_PARAMS = {
    "objective": "binary", "metric": "auc",
    "learning_rate": 0.03, "num_leaves": 127, "min_child_samples": 30,
    "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1,
    "lambda_l1": 0.1, "lambda_l2": 0.5,
    "verbosity": -1, "seed": 42, "n_jobs": -1,
}

V1_XGB_PARAMS = {
    "objective": "binary:logistic", "eval_metric": "auc",
    "learning_rate": 0.03, "max_depth": 7, "min_child_weight": 30,
    "subsample": 0.8, "colsample_bytree": 0.8,
    "reg_alpha": 0.1, "reg_lambda": 0.5,
    "tree_method": "hist", "random_state": 42, "n_jobs": -1, "verbosity": 0,
}

V1_CAT_PARAMS = {
    "iterations": 10000,
    "learning_rate": 0.020277240975450236,
    "depth": 8,
    "l2_leaf_reg": 6.347428100049705,
    "bagging_temperature": 0.6030591408625682,
    "random_strength": 1.641591378496605,
    "min_data_in_leaf": 42,
    "eval_metric": "AUC", "loss_function": "Logloss",
    "random_seed": 42, "verbose": False, "od_type": "Iter", "od_wait": 200,
    "thread_count": -1,
}

print("="*60)
print("v1 학습 (default LGBM/XGB + Optuna CAT)")
print("="*60)
t_v1 = time.time()
v1_oof_lgb, v1_oof_xgb, v1_oof_cat, v1_test_lgb, v1_test_xgb, v1_test_cat = (
    train_gbm_5fold(
        X, y, X_test, X_xgb, X_test_xgb,
        V1_LGBM_PARAMS, V1_XGB_PARAMS, V1_CAT_PARAMS,
        te_cols=te_cols_present, n_splits=N_SPLITS, seed=SEED, label="v1"
    )
)
print(f"v1 총 소요: {(time.time()-t_v1)/60:.1f} min")

# 원본 train_v1_optunaCAT.py와 동일한 파일명으로 저장 → blend 호환
np.savez(MODELS / "oof_v1_optunaCAT.npz",
         lgbm=v1_oof_lgb, xgb=v1_oof_xgb, cat=v1_oof_cat, y=y.values)
np.savez(MODELS / "test_v1_optunaCAT.npz",
         lgbm=v1_test_lgb, xgb=v1_test_xgb, cat=v1_test_cat)
print("✅ models/oof_v1_optunaCAT.npz, models/test_v1_optunaCAT.npz")


v1 학습 (default LGBM/XGB + Optuna CAT)
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[160]	valid_0's auc: 0.737578
  [v1] Fold 1: LGBM=0.73758  XGB=0.73832  CAT=0.73846  (312s)
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[191]	valid_0's auc: 0.74203
  [v1] Fold 2: LGBM=0.74203  XGB=0.74263  CAT=0.74328  (377s)
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[106]	valid_0's auc: 0.739658
  [v1] Fold 3: LGBM=0.73966  XGB=0.74044  CAT=0.74079  (520s)
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[118]	valid_0's auc: 0.73794
  [v1] Fold 4: LGBM=0.73794  XGB=0.73841  CAT=0.73846  (339s)
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[114]	valid_0's auc: 0.739829
  [v1] Fold 5: LGBM=0.73983  XGB=0.74108  CAT=0.74091  (369s)
  [v1] OOF  LGBM=0.

## 7. v2 모델 학습 — `train_v2_full.py`

3개 GBM 모두 Optuna 결과 적용:
- LightGBM Optuna: `tune_optuna_lgbm.py` 결과 (Appendix A)
- XGBoost Optuna: `tune_optuna_xgb.py` 결과 (Appendix A)
- CatBoost Optuna: `tune_optuna.py` 결과 (= V1_CAT_PARAMS와 동일)

저장: `models/oof_v2_ALL_OPTUNA.npz`, `models/test_v2_ALL_OPTUNA.npz` (blend 호환)

> **기대 OOF**: lgb ≈ 0.7402, xgb ≈ 0.7404, cat ≈ 0.7404 (PDF p.20)

In [10]:
# train_v2_full.py 원본 파라미터 (Optuna best params)
V2_LGBM_PARAMS = {
    "objective": "binary", "metric": "auc",
    "learning_rate": 0.03808978496858034,
    "num_leaves": 164,
    "max_depth": 5,
    "min_child_samples": 65,
    "feature_fraction": 0.6682096494749166,
    "bagging_fraction": 0.6260206371941118,
    "bagging_freq": 7,
    "lambda_l1": 1.540211302189079,
    "lambda_l2": 0.46616954181056797,
    "verbosity": -1, "seed": 42, "n_jobs": -1,
}

V2_XGB_PARAMS = {
    "objective": "binary:logistic", "eval_metric": "auc",
    "learning_rate": 0.025250437590382042,
    "max_depth": 5,
    "min_child_weight": 75,
    "subsample": 0.7659271582233903,
    "colsample_bytree": 0.7996037522585402,
    "reg_alpha": 0.3015538279990218,
    "reg_lambda": 1.1054754050726803,
    "gamma": 0.27694592828427284,
    "tree_method": "hist", "random_state": 42, "n_jobs": -1, "verbosity": 0,
}

V2_CAT_PARAMS = V1_CAT_PARAMS  # 원본과 동일 (PDF p.23)

print("="*60)
print("v2 학습 (Optuna LGBM + Optuna XGB + Optuna CAT)")
print("="*60)
t_v2 = time.time()
v2_oof_lgb, v2_oof_xgb, v2_oof_cat, v2_test_lgb, v2_test_xgb, v2_test_cat = (
    train_gbm_5fold(
        X, y, X_test, X_xgb, X_test_xgb,
        V2_LGBM_PARAMS, V2_XGB_PARAMS, V2_CAT_PARAMS,
        te_cols=te_cols_present, n_splits=N_SPLITS, seed=SEED, label="v2"
    )
)
print(f"v2 총 소요: {(time.time()-t_v2)/60:.1f} min")

# blend_8model_tabnet.py가 참조하는 정확한 파일명으로 저장
np.savez(MODELS / "oof_v2_ALL_OPTUNA.npz",
         lgbm=v2_oof_lgb, xgb=v2_oof_xgb, cat=v2_oof_cat, y=y.values)
np.savez(MODELS / "test_v2_ALL_OPTUNA.npz",
         lgbm=v2_test_lgb, xgb=v2_test_xgb, cat=v2_test_cat)
# 호환성: oof_v2.npz도 함께 저장 (train_v2_full.py 원본 파일명)
np.savez(MODELS / "oof_v2.npz",
         lgbm=v2_oof_lgb, xgb=v2_oof_xgb, cat=v2_oof_cat, y=y.values)
np.savez(MODELS / "test_v2.npz",
         lgbm=v2_test_lgb, xgb=v2_test_xgb, cat=v2_test_cat)
print("✅ models/oof_v2_ALL_OPTUNA.npz, models/test_v2_ALL_OPTUNA.npz")


v2 학습 (Optuna LGBM + Optuna XGB + Optuna CAT)
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[310]	valid_0's auc: 0.738222
  [v2] Fold 1: LGBM=0.73822  XGB=0.73819  CAT=0.73846  (318s)
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[513]	valid_0's auc: 0.742397
  [v2] Fold 2: LGBM=0.74240  XGB=0.74303  CAT=0.74328  (383s)
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[383]	valid_0's auc: 0.740725
  [v2] Fold 3: LGBM=0.74073  XGB=0.74066  CAT=0.74079  (522s)
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[259]	valid_0's auc: 0.738371
  [v2] Fold 4: LGBM=0.73837  XGB=0.73861  CAT=0.73846  (336s)
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[392]	valid_0's auc: 0.741451
  [v2] Fold 5: LGBM=0.74145  XGB=0.74160  CAT=0.74091  (348s)
  [v2] OO

## 8. MLP — `train_mlp.py`

PyTorch MLP, 5-fold OOF.
- Architecture: Embedding + [256, 128, 64] + BatchNorm + Dropout
- Mac M2 MPS / CUDA / CPU 자동 감지
- 약 8-10분 (M2 Pro 기준)

> **기대 OOF**: ≈ 0.7393 (PDF p.20)

In [ ]:
# train_mlp.py 원본 그대로 inline
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Device 자동 선택
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Device: {device}", flush=True)
torch.manual_seed(42)
np.random.seed(42)

# §5의 X, X_test, y 재활용 (X에는 TE 없음, X_test에는 TE 있음 → 제거)
mlp_X      = X.copy()
mlp_X_test = X_test[list(X.columns)].copy()  # TE 컬럼 제거
mlp_y      = y.values

# train_mlp.py 원본의 is_cat_col
def is_cat_col(s):
    if s.dtype == 'object' or pd.api.types.is_string_dtype(s):
        return True
    if isinstance(s.dtype, pd.CategoricalDtype):
        return True
    if str(s.dtype).startswith('string'):
        return True
    sample = s.dropna().head(50)
    if len(sample) == 0:
        return False
    try:
        pd.to_numeric(sample); return False
    except (ValueError, TypeError):
        return True

obj_cols_mlp = [c for c in mlp_X.columns if is_cat_col(mlp_X[c])]
num_cols_mlp = [c for c in mlp_X.columns if c not in obj_cols_mlp]
print(f"  Categorical: {len(obj_cols_mlp)}, Numerical: {len(num_cols_mlp)}")

# Label encode (NaN-safe)
cat_dim_list_mlp = []
for c in obj_cols_mlp:
    arr_X = mlp_X[c].values
    arr_test = mlp_X_test[c].values
    def to_safe_str(arr):
        out = np.empty(len(arr), dtype=object)
        for i, v in enumerate(arr):
            if v is None or (isinstance(v, float) and np.isnan(v)) or (isinstance(v, str) and v == 'nan'):
                out[i] = '__NA__'
            else:
                out[i] = str(v)
        return out
    s_X    = to_safe_str(arr_X)
    s_test = to_safe_str(arr_test)
    all_vals = np.concatenate([s_X, s_test])
    uniq = sorted(set(all_vals.tolist()))
    mapping = {v: i for i, v in enumerate(uniq)}
    mlp_X[c]      = np.array([mapping[v] for v in s_X], dtype=np.int32)
    mlp_X_test[c] = np.array([mapping[v] for v in s_test], dtype=np.int32)
    cat_dim_list_mlp.append(len(uniq))

# Numerical fill
for c in num_cols_mlp:
    mlp_X[c]      = pd.to_numeric(mlp_X[c],      errors='coerce').fillna(0).astype("float32")
    mlp_X_test[c] = pd.to_numeric(mlp_X_test[c], errors='coerce').fillna(0).astype("float32")

X_num_all  = mlp_X[num_cols_mlp].values.astype("float32")
X_cat_all  = mlp_X[obj_cols_mlp].values.astype("int64")
X_test_num = mlp_X_test[num_cols_mlp].values.astype("float32")
X_test_cat = mlp_X_test[obj_cols_mlp].values.astype("int64")
print(f"  X_num={X_num_all.shape}, X_cat={X_cat_all.shape}")


class TabMLP(nn.Module):
    def __init__(self, num_n, cat_dims, emb_dim=8, hidden=[256,128,64], dropout=0.3):
        super().__init__()
        self.embs = nn.ModuleList([
            nn.Embedding(d, min(emb_dim, max(2, d // 2 + 1))) for d in cat_dims
        ])
        emb_total = sum(min(emb_dim, max(2, d // 2 + 1)) for d in cat_dims)
        in_dim = num_n + emb_total
        layers = []
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.mlp = nn.Sequential(*layers)
    def forward(self, x_num, x_cat):
        embs = [e(x_cat[:, i]) for i, e in enumerate(self.embs)]
        z = torch.cat([x_num] + embs, dim=1)
        return self.mlp(z).squeeze(-1)


# Training
N_FOLDS = 5
EPOCHS  = 30
BATCH   = 4096
LR      = 1e-3
WD      = 1e-5
PATIENCE = 5

skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=42)
oof_mlp  = np.zeros(len(mlp_X))
test_mlp = np.zeros(len(mlp_X_test))

print(f"\n[2/4] Training (epochs={EPOCHS}, batch={BATCH})", flush=True)
t0 = time.time()
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_num_all, mlp_y)):
    ts = time.time()

    scaler = StandardScaler()
    Xtr_num = scaler.fit_transform(X_num_all[tr_idx])
    Xva_num = scaler.transform(X_num_all[va_idx])
    Xte_num = scaler.transform(X_test_num)

    Xtr_num_t = torch.from_numpy(Xtr_num.astype("float32"))
    Xva_num_t = torch.from_numpy(Xva_num.astype("float32"))
    Xte_num_t = torch.from_numpy(Xte_num.astype("float32"))
    Xtr_cat_t = torch.from_numpy(X_cat_all[tr_idx])
    Xva_cat_t = torch.from_numpy(X_cat_all[va_idx])
    Xte_cat_t = torch.from_numpy(X_test_cat)
    ytr_t = torch.from_numpy(mlp_y[tr_idx].astype("float32"))

    train_ds = TensorDataset(Xtr_num_t, Xtr_cat_t, ytr_t)
    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=0)

    model = TabMLP(num_n=Xtr_num.shape[1], cat_dims=cat_dim_list_mlp).to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=EPOCHS)
    bce = nn.BCEWithLogitsLoss()

    best_auc, best_va, best_te, no_improve = 0, None, None, 0
    for ep in range(EPOCHS):
        model.train()
        for xn, xc, yb in train_loader:
            xn, xc, yb = xn.to(device), xc.to(device), yb.to(device)
            optim.zero_grad()
            loss = bce(model(xn, xc), yb)
            loss.backward()
            optim.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            va_logit = model(Xva_num_t.to(device), Xva_cat_t.to(device)).cpu().numpy()
        va_pred = 1 / (1 + np.exp(-va_logit))
        auc = roc_auc_score(mlp_y[va_idx], va_pred)
        if auc > best_auc:
            best_auc = auc
            best_va = va_pred
            with torch.no_grad():
                te_logit = model(Xte_num_t.to(device), Xte_cat_t.to(device)).cpu().numpy()
            best_te = 1 / (1 + np.exp(-te_logit))
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                break

    oof_mlp[va_idx] = best_va
    test_mlp += best_te / N_FOLDS
    print(f"  Fold {fold+1}: AUC={best_auc:.5f}  best_ep={ep-no_improve+1}/{ep+1}  "
          f"({time.time()-ts:.0f}s)", flush=True)

print(f"\n[3/4] OOF AUC: {roc_auc_score(mlp_y, oof_mlp):.5f}")
np.savez_compressed(MODELS / "oof_mlp.npz",  mlp=oof_mlp, y=mlp_y)
np.savez_compressed(MODELS / "test_mlp.npz", mlp=test_mlp)
print(f"\n[4/4] Done. Total: {(time.time()-t0)/60:.1f} min")
print(f"  Saved: models/oof_mlp.npz, models/test_mlp.npz")


Device: mps
  Categorical: 23, Numerical: 187
  X_num=(256351, 187), X_cat=(256351, 23)

[2/4] Training (epochs=30, batch=4096)


## 9. TabNet — `train_tabnet.py`

Dreamquark TabNet (attention-based feature selection).
- 5-fold OOF
- 약 30-50분 (M2 Pro 기준)

> **기대 OOF**: ≈ 0.7375 (PDF p.20)

In [ ]:
# train_tabnet.py 원본 그대로
from pytorch_tabnet.tab_model import TabNetClassifier

device_name = "cuda" if torch.cuda.is_available() else (
    "mps" if (hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()) else "cpu"
)
print(f"Device: {device_name}", flush=True)
torch.manual_seed(456)
np.random.seed(456)

# §8의 mlp_X, mlp_X_test에 이미 label-encoded 되어 있으므로 그대로 사용
# (TabNet은 단일 매트릭스를 받음)
all_cols_tn = list(mlp_X.columns)
cat_idxs    = [all_cols_tn.index(c) for c in obj_cols_mlp]

X_full_tn      = mlp_X[all_cols_tn].values.astype("float32")
X_test_full_tn = mlp_X_test[all_cols_tn].values.astype("float32")
print(f"  X={X_full_tn.shape}, X_test={X_test_full_tn.shape}")
print(f"  cat_idxs={len(cat_idxs)}, cat_dims={cat_dim_list_mlp[:5]}...")

# Training
MAX_EPOCHS = 100
PATIENCE_TN = 10
BATCH = 4096
VIRTUAL_BATCH = 512

skf_tn = StratifiedKFold(N_FOLDS, shuffle=True, random_state=42)
oof_tabnet  = np.zeros(len(mlp_X))
test_tabnet = np.zeros(len(mlp_X_test))

print(f"\n[2/4] TabNet training", flush=True)
t0 = time.time()
for fold, (tr_idx, va_idx) in enumerate(skf_tn.split(X_full_tn, mlp_y)):
    ts = time.time()

    scaler = StandardScaler()
    Xtr = X_full_tn[tr_idx].copy()
    Xva = X_full_tn[va_idx].copy()
    Xte = X_test_full_tn.copy()
    num_idxs = [i for i in range(X_full_tn.shape[1]) if i not in cat_idxs]
    Xtr[:, num_idxs] = scaler.fit_transform(Xtr[:, num_idxs])
    Xva[:, num_idxs] = scaler.transform(Xva[:, num_idxs])
    Xte[:, num_idxs] = scaler.transform(Xte[:, num_idxs])

    model = TabNetClassifier(
        n_d=32, n_a=32, n_steps=4,
        gamma=1.3, n_independent=2, n_shared=2,
        cat_idxs=cat_idxs,
        cat_dims=cat_dim_list_mlp,
        cat_emb_dim=8,
        seed=42 + fold,
        device_name=device_name,
        verbose=0,
        optimizer_params=dict(lr=2e-2),
        scheduler_fn=torch.optim.lr_scheduler.CosineAnnealingLR,
        scheduler_params={"T_max": MAX_EPOCHS},
        mask_type='entmax',
    )
    model.fit(
        X_train=Xtr, y_train=mlp_y[tr_idx],
        eval_set=[(Xva, mlp_y[va_idx])],
        eval_metric=['auc'],
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE_TN,
        batch_size=BATCH, virtual_batch_size=VIRTUAL_BATCH,
        drop_last=False,
    )
    va_pred = model.predict_proba(Xva)[:, 1]
    te_pred = model.predict_proba(Xte)[:, 1]
    auc = roc_auc_score(mlp_y[va_idx], va_pred)
    oof_tabnet[va_idx] = va_pred
    test_tabnet += te_pred / N_FOLDS
    print(f"  Fold {fold+1}: AUC={auc:.5f}  best_ep={model.best_epoch}  "
          f"({time.time()-ts:.0f}s)", flush=True)

print(f"\n[3/4] OOF AUC: {roc_auc_score(mlp_y, oof_tabnet):.5f}")
np.savez_compressed(MODELS / "oof_tabnet.npz",  tabnet=oof_tabnet, y=mlp_y)
np.savez_compressed(MODELS / "test_tabnet.npz", tabnet=test_tabnet)
print(f"\n[4/4] Done. Total: {(time.time()-t0)/60:.1f} min")
print(f"  Saved: models/oof_tabnet.npz, models/test_tabnet.npz")


## 10. 8-Model Blending — `blend_8model_tabnet.py`

원본 그대로. npz 파일에서 로드 → Nelder-Mead → submission.

> **기대 결과**: 8-blend OOF ≈ **0.74101** (PDF p.25, p.31)
> 예상 LB ≈ 0.74235 (3rd place)

In [ ]:
"""8-blend: 6 GBM + MLP + TabNet"""

v1_oof = np.load(MODELS / 'oof_v1_optunaCAT.npz')
v1_test = np.load(MODELS / 'test_v1_optunaCAT.npz')
v2_oof = np.load(MODELS / 'oof_v2_ALL_OPTUNA.npz')
v2_test = np.load(MODELS / 'test_v2_ALL_OPTUNA.npz')
mlp_oof = np.load(MODELS / 'oof_mlp.npz')
mlp_test = np.load(MODELS / 'test_mlp.npz')
tn_oof = np.load(MODELS / 'oof_tabnet.npz')
tn_test = np.load(MODELS / 'test_tabnet.npz')
y = v1_oof['y']

oofs = np.column_stack([
    v1_oof['lgbm'], v1_oof['xgb'], v1_oof['cat'],
    v2_oof['lgbm'], v2_oof['xgb'], v2_oof['cat'],
    mlp_oof['mlp'], tn_oof['tabnet'],
]).astype(np.float32)

tests = np.column_stack([
    v1_test['lgbm'], v1_test['xgb'], v1_test['cat'],
    v2_test['lgbm'], v2_test['xgb'], v2_test['cat'],
    mlp_test['mlp'], tn_test['tabnet'],
]).astype(np.float32)

names = ['v1_lgb', 'v1_xgb', 'v1_cat', 'v2_lgb', 'v2_xgb', 'v2_cat', 'mlp', 'tabnet']

print('=' * 60)
print('Individual OOF AUC:')
for i, n in enumerate(names):
    print(f'  {n}: {roc_auc_score(y, oofs[:, i]):.5f}')

print('\nTabNet correlation analysis:')
print(f'  tabnet vs mlp:    {np.corrcoef(oofs[:,7], oofs[:,6])[0,1]:.4f}')
print(f'  tabnet vs v1_lgb: {np.corrcoef(oofs[:,7], oofs[:,0])[0,1]:.4f}')
print(f'  tabnet vs v1_cat: {np.corrcoef(oofs[:,7], oofs[:,2])[0,1]:.4f}')
print(f'  tabnet vs v2_xgb: {np.corrcoef(oofs[:,7], oofs[:,4])[0,1]:.4f}')

def neg_auc(w):
    if (w < 0).any() or w.sum() < 0.99: return 0
    return -roc_auc_score(y, oofs @ (w / w.sum()))

best, bw = 0, None
for s in [0, 42, 7, 123, 999]:
    rng = np.random.default_rng(s)
    x0 = rng.dirichlet(np.ones(8))
    res = minimize(neg_auc, x0, method='Nelder-Mead',
                   options={'xatol': 1e-5, 'fatol': 1e-7, 'maxiter': 1000})
    w = res.x / res.x.sum()
    auc = -res.fun
    if auc > best:
        best, bw = auc, w

print()
print('=' * 60)
print(f'7-blend (GBM+MLP):       0.74096  (예상 LB ~0.74247)')
print(f'8-blend (+TabNet):       {best:.5f}  (예상 LB ~{best + 0.00151:.5f})')
print(f'Δ vs 7-blend:             {best - 0.74096:+.5f}')
print('Weights:')
for n, w in zip(names, bw):
    print(f'  {n}: {w:.3f}')

test_blend = tests @ bw
sub = pd.read_csv(DATA / 'sample_submission.csv')
out = pd.DataFrame({'ID': sub['ID'], 'probability': test_blend})
out.to_csv(SUB / 'submission_8blend_tabnet.csv', index=False)

print()
print(f'Pred range: [{test_blend.min():.4f}, {test_blend.max():.4f}], mean={test_blend.mean():.4f}')
print(f'\n비교 (1등 0.74246):')
print(f'  7-blend:  0.74096 → ~0.74247 (+0.00001)')
print(f'  8-blend:  {best:.5f} → ~{best + 0.00151:.5f} ({best + 0.00151 - 0.74246:+.5f})')
print(f'\nSaved: submission/submission_8blend_tabnet.csv')


## 11. Data Leakage 종합 검증 (8/8 PASS 목표)

PDF p.32의 8가지 자동 검사 항목.

In [ ]:
print("="*60)
print("Data Leakage Audit")
print("="*60)

checks = []

# 01. Target column NOT in test
ok = TARGET not in pd.read_csv(DATA / "test.csv").columns
checks.append(("01 Target column NOT in test", ok))

# 02. No train-test ID overlap
tr_ids = pd.read_csv(DATA / "train.csv", usecols=[ID_COL])[ID_COL]
te_ids = pd.read_csv(DATA / "test.csv",  usecols=[ID_COL])[ID_COL]
ok = len(set(tr_ids) & set(te_ids)) == 0
checks.append(("02 No train-test ID overlap", ok))

# 03. v2 features deterministic (row order independent)
sample = pd.read_csv(DATA / "train.csv", nrows=100)
out_a = add_v2_all_features(sample.copy())
out_b = add_v2_all_features(sample.iloc[::-1].copy()).iloc[::-1]
common_cols = [c for c in out_a.columns if c in out_b.columns
               and pd.api.types.is_numeric_dtype(out_a[c])]
ok = all(np.allclose(out_a[c].fillna(-999), out_b[c].fillna(-999))
         for c in common_cols[:50])
checks.append(("03 v2 features row-deterministic", ok))

# 04. OOFTargetEncoder has both methods (fold-internal + full-train)
ok = (hasattr(OOFTargetEncoder, 'fit_transform_oof')
      and hasattr(OOFTargetEncoder, 'fit_full_then_transform'))
checks.append(("04 OOFTargetEncoder has both methods", ok))

# 05. No test labels used (test 자체에 target 없음)
ok = TARGET not in X_test.columns
checks.append(("05 Test set has NO target column", ok))

# 06. Submission sanity
sub_check = pd.read_csv(SUB / "submission_8blend_tabnet.csv")
ok = (sub_check['probability'].between(0, 1).all()
      and sub_check['probability'].isna().sum() == 0
      and 0.10 < sub_check['probability'].mean() < 0.50)
checks.append(("06 Submission [0,1], NaN-free, sensible mean", ok))

# 07. All 8 models same length
ok = (len(v1_oof_lgb) == len(v2_oof_lgb) == len(oof_mlp) == len(oof_tabnet)
      == len(y))
checks.append(("07 All 8 models share same len(y)", ok))

# 08. Blend weights ≥0, Σ=1
ok = (bw >= 0).all() and abs(bw.sum() - 1.0) < 1e-6
checks.append(("08 Blend weights w>=0, sum=1", ok))

n_pass = sum(1 for _, ok in checks if ok)
print(f"\n{n_pass}/{len(checks)} PASS\n")
for label, ok in checks:
    print(f"  {'✅' if ok else '❌'} {label}")

if n_pass == len(checks):
    print("\n🎉 ALL CHECKS PASSED — submission is leakage-safe")
else:
    print("\n⚠️  FAIL 항목 확인 필요")


## 12. 최종 결과 요약

In [ ]:
y_arr = np.asarray(y)

print("="*60)
print("🏆 IVF Pregnancy Prediction — Final Results")
print("="*60)
print(f"\nDataset      : {len(y):,} rows × {X.shape[1]} features (after v2 + TE)")
print(f"Models       : 8 (6 GBM + MLP + TabNet)")
print(f"CV scheme    : StratifiedKFold(5, shuffle=True, random_state={SEED})")

print(f"\nIndividual OOF AUC:")
for i, n in enumerate(names):
    print(f"  {n:8s} : {roc_auc_score(y_arr, oofs[:, i]):.5f}")

print(f"\n8-blend OOF AUC : {best:.5f}")
print(f"PDF reported    : 0.74101 (OOF) → 0.74235 (Public LB, 3rd place)")

# Save final summary
summary_path = MODELS / "final_summary.json"
summary_path.write_text(json.dumps({
    "individual_oof_aucs": {n: float(roc_auc_score(y_arr, oofs[:, i]))
                            for i, n in enumerate(names)},
    "blend_oof_auc": float(best),
    "blend_weights": {n: float(w) for n, w in zip(names, bw)},
    "n_train_rows": int(len(y)),
    "n_features": int(X.shape[1]),
    "n_splits": int(N_SPLITS),
    "seed": int(SEED),
}, ensure_ascii=False, indent=2))
print(f"\nSubmission   : submission/submission_8blend_tabnet.csv")
print(f"Summary      : {summary_path.relative_to(ROOT)}")
print("\n✅ 실행 완료")


---

# 📎 Appendix A — Optuna Hyperparameter Tuning *(선택 실행)*

§6, §7의 GBM 파라미터는 **이미 도출된 Optuna best params**를 hardcoded로 사용했습니다.
이 부록은 **Optuna 탐색 과정 자체를 재현**하고 싶을 때만 실행하세요.

> **⚠️ 주의**: 실행 시간이 매우 큽니다.
> - CatBoost: 30-60분 (30 trials × 3-fold)
> - LightGBM: 19분
> - XGBoost: 21분
> - **합계 약 60-100분 추가**

이 셀은 **TPESampler(seed=42) 결과가 결정론적**이므로,
이미 §6, §7에서 사용한 hardcoded params를 다시 확인하는 용도로만 사용됩니다.

### A.1 CatBoost Optuna — `tune_optuna.py`

In [ ]:
# CatBoost Optuna - 결과 hardcoded되어 있어 실행 불필요
# 실행하려면 아래 셀 주석 해제

# !pip install optuna  # 미설치 시
# import optuna
# from optuna.samplers import TPESampler
"""
tune_optuna.py — Optuna hyperparameter search for CatBoost (best single model)
==================================================================================
Reasoning: CatBoost is best individual model (typically AUC 0.7401-0.7415 range).
Optuna search over depth/leaf_reg/lr/random_strength can push another +0.001-0.003.

Search space (informed by literature for IVF-style tabular):
  - depth: 6-9
  - learning_rate: 0.02-0.07
  - l2_leaf_reg: 1-10
  - random_strength: 0.5-2.0
  - bagging_temperature: 0.3-1.5
  - subsample: 0.7-0.95
  - iterations: 5000 fixed (with early stopping)

Run-time: ~30-60 minutes for 30 trials on full data.
For faster iteration: use --trials 15 --subsample-rate 0.5 (50% sample).

VS Code:
    pip install optuna
    python src/tune_optuna.py --trials 30
"""





def evaluate_params(params: dict, X: pd.DataFrame, y: pd.Series, n_folds: int = 3) -> float:
    """3-fold OOF AUC (faster than 5-fold for tuning)."""
    skf = StratifiedKFold(n_folds, shuffle=True, random_state=42)
    enc = OOFTargetEncoder(cols=[c for c in TARGET_ENC_COLS if c in X.columns], smoothing=50.0)
    oof = np.zeros(len(X))

    for tr, va in skf.split(X, y):
        Xtr_b = X.iloc[tr].reset_index(drop=True)
        Xva_b = X.iloc[va].reset_index(drop=True)
        ytr = y.iloc[tr].reset_index(drop=True)
        yva = y.iloc[va].reset_index(drop=True)

        tr_te, va_te = enc.fit_transform_oof(Xtr_b, ytr, Xva_b)
        Xtr = pd.concat([Xtr_b, tr_te], axis=1)
        Xva = pd.concat([Xva_b, va_te], axis=1)

        cat_features = Xtr.select_dtypes(include=["object", "category"]).columns.tolist()
        Xtr = prep_for_cat(Xtr, cat_features)
        Xva = prep_for_cat(Xva, cat_features)

        m = CatBoostClassifier(**params)
        m.fit(Pool(Xtr, ytr, cat_features=cat_features),
              eval_set=Pool(Xva, yva, cat_features=cat_features),
              use_best_model=True, verbose=False)
        oof[va] = m.predict_proba(Pool(Xva, cat_features=cat_features))[:, 1]

    return roc_auc_score(y, oof)


def main(n_trials: int = 30, subsample_rate: float = 1.0, n_folds: int = 3):
    t0 = time.time()
    print(f"[1/3] Load data + v2 features", flush=True)
    train = pd.read_csv(DATA / "train.csv")
    if subsample_rate < 1.0:
        train = train.sample(frac=subsample_rate, random_state=42).reset_index(drop=True)
        print(f"  Subsampled to {len(train)} rows", flush=True)

    train = add_v2_all_features(train)
    y = train[TARGET].astype(int).reset_index(drop=True)
    drop_cols = [TARGET, ID_COL] if ID_COL in train.columns else [TARGET]
    X = train.drop(columns=drop_cols).reset_index(drop=True)
    print(f"  X: {X.shape}", flush=True)

    print(f"\n[2/3] Optuna search ({n_trials} trials, {n_folds}-fold OOF)", flush=True)

    def objective(trial):
        params = {
            "iterations": 5000,
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.07, log=True),
            "depth": trial.suggest_int("depth", 5, 9),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0, log=True),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.3, 1.5),
            "random_strength": trial.suggest_float("random_strength", 0.5, 2.0),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 100),
            "eval_metric": "AUC", "loss_function": "Logloss",
            "random_seed": 42, "verbose": False,
            "od_type": "Iter", "od_wait": 150, "thread_count": -1,
        }
        try:
            auc = evaluate_params(params, X, y, n_folds=n_folds)
            return auc
        except Exception as e:
            print(f"  Trial failed: {e}")
            return 0.0

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    print(f"\n[3/3] Best: {study.best_value:.5f}", flush=True)
    print(f"  Best params: {study.best_params}", flush=True)
    print(f"  Total time: {(time.time()-t0)/60:.1f} min", flush=True)

    summary = {
        "best_auc": float(study.best_value),
        "best_params": study.best_params,
        "n_trials": n_trials,
        "n_folds": n_folds,
        "subsample_rate": subsample_rate,
        "all_trial_aucs": [t.value for t in study.trials if t.value is not None],
        "elapsed_min": round((time.time() - t0) / 60, 1),
    }
    (MODELS / "optuna_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2))
    print(f"  Saved: models/optuna_summary.json")
    print(f"\n  Use these params in src/train_v2_full.py CAT_PARAMS dict, then re-run training.")

# main(n_trials=30, subsample_rate=1.0, n_folds=3)


### A.2 LightGBM Optuna — `tune_optuna_lgbm.py`

In [ ]:
# LightGBM Optuna - 결과 hardcoded되어 있어 실행 불필요
# import optuna
# from optuna.samplers import TPESampler
"""
tune_optuna_lgbm.py — Optuna for LightGBM
Search space:
  - num_leaves: 31-255
  - learning_rate: 0.02-0.07
  - min_child_samples: 10-100
  - feature_fraction: 0.6-1.0
  - bagging_fraction: 0.6-1.0
  - lambda_l1/l2: 0.0-2.0
"""


def prep_for_lgb(X):
    """LGBM accepts category dtype natively."""
    X = X.copy()
    obj_cols = X.select_dtypes(include=["object"]).columns.tolist()
    for c in obj_cols:
        X[c] = X[c].astype("category")
    return X, obj_cols

def evaluate_params(params: dict, X: pd.DataFrame, y: pd.Series, n_folds: int = 3) -> float:
    skf = StratifiedKFold(n_folds, shuffle=True, random_state=42)
    enc = OOFTargetEncoder(cols=[c for c in TARGET_ENC_COLS if c in X.columns], smoothing=50.0)
    oof = np.zeros(len(X))
    for tr, va in skf.split(X, y):
        Xtr_b = X.iloc[tr].reset_index(drop=True)
        Xva_b = X.iloc[va].reset_index(drop=True)
        ytr = y.iloc[tr].reset_index(drop=True)
        yva = y.iloc[va].reset_index(drop=True)
        tr_te, va_te = enc.fit_transform_oof(Xtr_b, ytr, Xva_b)
        Xtr = pd.concat([Xtr_b, tr_te], axis=1)
        Xva = pd.concat([Xva_b, va_te], axis=1)
        Xtr, cat_cols = prep_for_lgb(Xtr)
        Xva, _ = prep_for_lgb(Xva)
        ds_tr = lgb.Dataset(Xtr, label=ytr, categorical_feature=cat_cols)
        ds_va = lgb.Dataset(Xva, label=yva, categorical_feature=cat_cols, reference=ds_tr)
        m = lgb.train(params, ds_tr, num_boost_round=5000,
                      valid_sets=[ds_va],
                      callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(0)])
        oof[va] = m.predict(Xva, num_iteration=m.best_iteration)
    return roc_auc_score(y, oof)

def main(n_trials: int = 30, subsample_rate: float = 1.0, n_folds: int = 3):
    t0 = time.time()
    print(f"[1/3] Load data + v2 features (LGBM Optuna)", flush=True)
    train = pd.read_csv(DATA / "train.csv")
    if subsample_rate < 1.0:
        train = train.sample(frac=subsample_rate, random_state=42).reset_index(drop=True)
        print(f"  Subsampled to {len(train)} rows", flush=True)
    train = add_v2_all_features(train)
    y = train[TARGET].astype(int).reset_index(drop=True)
    drop_cols = [TARGET, ID_COL] if ID_COL in train.columns else [TARGET]
    X = train.drop(columns=drop_cols).reset_index(drop=True)
    print(f"  X: {X.shape}", flush=True)

    print(f"\n[2/3] Optuna search ({n_trials} trials, {n_folds}-fold OOF)", flush=True)
    def objective(trial):
        params = {
            "objective": "binary", "metric": "auc",
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.07, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 31, 255),
            "max_depth": trial.suggest_int("max_depth", 5, 12),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
            "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
            "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
            "lambda_l1": trial.suggest_float("lambda_l1", 1e-3, 2.0, log=True),
            "lambda_l2": trial.suggest_float("lambda_l2", 1e-3, 2.0, log=True),
            "verbosity": -1, "seed": 42, "n_jobs": -1,
        }
        try:
            auc = evaluate_params(params, X, y, n_folds=n_folds)
            return auc
        except Exception as e:
            print(f"  Trial failed: {e}")
            return 0.0

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    print(f"\n[3/3] Best: {study.best_value:.5f}", flush=True)
    print(f"  Best params: {study.best_params}", flush=True)
    print(f"  Total time: {(time.time()-t0)/60:.1f} min", flush=True)

    summary = {
        "model": "lgbm",
        "best_auc": float(study.best_value),
        "best_params": study.best_params,
        "n_trials": n_trials,
        "n_folds": n_folds,
        "subsample_rate": subsample_rate,
        "all_trial_aucs": [t.value for t in study.trials if t.value is not None],
        "elapsed_min": round((time.time() - t0) / 60, 1),
    }
    (MODELS / "optuna_summary_lgbm.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2))
    print(f"  Saved: models/optuna_summary_lgbm.json")

# main(n_trials=30, subsample_rate=1.0, n_folds=3)


### A.3 XGBoost Optuna — `tune_optuna_xgb.py`

In [ ]:
# XGBoost Optuna - 결과 hardcoded되어 있어 실행 불필요
# import optuna
# from optuna.samplers import TPESampler
"""
tune_optuna_xgb.py — Optuna for XGBoost
"""


def build_universe(X_, X_test_, cols):
    universe = {}
    for c in cols:
        s = pd.concat([X_[c], X_test_[c]]) if X_test_ is not None else X_[c]
        s = s.fillna('__NA__').astype(str)
        vals = sorted(s.unique().tolist())
        universe[c] = {v: i for i, v in enumerate(vals)}
    return universe

def encode_int(df, universe):
    df_e = df.copy()
    for c, mp in universe.items():
        s = df[c].astype(object)
        s = pd.Series(s).where(pd.notna(s), '__NA__').astype(str)
        df_e[c] = s.map(mp).fillna(-1).astype('int32').values
    return df_e

def evaluate_params(params: dict, X: pd.DataFrame, y: pd.Series, n_folds: int = 3) -> float:
    skf = StratifiedKFold(n_folds, shuffle=True, random_state=42)
    enc = OOFTargetEncoder(cols=[c for c in TARGET_ENC_COLS if c in X.columns], smoothing=50.0)
    oof = np.zeros(len(X))
    for tr, va in skf.split(X, y):
        Xtr_b = X.iloc[tr].reset_index(drop=True)
        Xva_b = X.iloc[va].reset_index(drop=True)
        ytr = y.iloc[tr].reset_index(drop=True)
        yva = y.iloc[va].reset_index(drop=True)
        tr_te, va_te = enc.fit_transform_oof(Xtr_b, ytr, Xva_b)
        Xtr = pd.concat([Xtr_b, tr_te], axis=1)
        Xva = pd.concat([Xva_b, va_te], axis=1)
        # Build int universe from train+val (OK since fold internal)
        obj_cols = Xtr.select_dtypes(include=["object", "category"]).columns.tolist()
        universe = build_universe(Xtr, Xva, obj_cols)
        Xtr_int = encode_int(Xtr, universe)
        Xva_int = encode_int(Xva, universe)
        dtr = xgb.DMatrix(Xtr_int, label=ytr)
        dva = xgb.DMatrix(Xva_int, label=yva)
        m = xgb.train(params, dtr, num_boost_round=5000,
                      evals=[(dva, "va")],
                      early_stopping_rounds=150, verbose_eval=False)
        oof[va] = m.predict(dva, iteration_range=(0, m.best_iteration + 1))
    return roc_auc_score(y, oof)

def main(n_trials: int = 30, subsample_rate: float = 1.0, n_folds: int = 3):
    t0 = time.time()
    print(f"[1/3] Load data + v2 features (XGB Optuna)", flush=True)
    train = pd.read_csv(DATA / "train.csv")
    if subsample_rate < 1.0:
        train = train.sample(frac=subsample_rate, random_state=42).reset_index(drop=True)
        print(f"  Subsampled to {len(train)} rows", flush=True)
    train = add_v2_all_features(train)
    y = train[TARGET].astype(int).reset_index(drop=True)
    drop_cols = [TARGET, ID_COL] if ID_COL in train.columns else [TARGET]
    X = train.drop(columns=drop_cols).reset_index(drop=True)
    print(f"  X: {X.shape}", flush=True)

    print(f"\n[2/3] Optuna search ({n_trials} trials, {n_folds}-fold OOF)", flush=True)
    def objective(trial):
        params = {
            "objective": "binary:logistic", "eval_metric": "auc",
            "tree_method": "hist",
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.07, log=True),
            "max_depth": trial.suggest_int("max_depth", 5, 10),
            "min_child_weight": trial.suggest_int("min_child_weight", 10, 100),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 2.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 2.0, log=True),
            "gamma": trial.suggest_float("gamma", 0.0, 1.0),
            "random_state": 42, "n_jobs": -1, "verbosity": 0,
        }
        try:
            auc = evaluate_params(params, X, y, n_folds=n_folds)
            return auc
        except Exception as e:
            print(f"  Trial failed: {e}")
            return 0.0

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    print(f"\n[3/3] Best: {study.best_value:.5f}", flush=True)
    print(f"  Best params: {study.best_params}", flush=True)
    print(f"  Total time: {(time.time()-t0)/60:.1f} min", flush=True)

    summary = {
        "model": "xgb",
        "best_auc": float(study.best_value),
        "best_params": study.best_params,
        "n_trials": n_trials,
        "n_folds": n_folds,
        "subsample_rate": subsample_rate,
        "all_trial_aucs": [t.value for t in study.trials if t.value is not None],
        "elapsed_min": round((time.time() - t0) / 60, 1),
    }
    (MODELS / "optuna_summary_xgb.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2))
    print(f"  Saved: models/optuna_summary_xgb.json")

# main(n_trials=30, subsample_rate=1.0, n_folds=3)


---

# 📎 Appendix B — FT-Transformer *(실험적, 8-blend 미포함)*

`train_ft_transformer.py` — Yandex 2021 FT-Transformer.

이 모델은 실험으로 학습되었으나 **최종 8-blend에는 포함되지 않았습니다.**
9-blend 또는 stacking 후속 실험을 위한 참고용 코드입니다.

> 약 30-60분 (M2 Pro)

In [ ]:
# FT-Transformer - 실험용 (8-blend 미포함). 실행 원하면 아래 코드 실행.

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# 주의: 아래 코드는 자체적으로 train.csv를 다시 로드합니다.
# (원본 train_ft_transformer.py 보존)

"""
FT-Transformer 5-fold OOF training.
Yandex 2021 — tabular SOTA architecture.
"""



device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}", flush=True)
torch.manual_seed(123)  # Different seed than MLP
np.random.seed(123)

# ─── Data prep (identical to MLP) ────────────────────────
print("[1/4] Load data + v2 features", flush=True)
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
train = add_v2_all_features(train)
test = add_v2_all_features(test)

y = train[TARGET].astype(int).values
X = train.drop(columns=[TARGET, ID_COL]).copy()
X_test = test.drop(columns=[ID_COL]).copy()

def is_cat_col(s):
    if s.dtype == 'object' or pd.api.types.is_string_dtype(s):
        return True
    if isinstance(s.dtype, pd.CategoricalDtype):
        return True
    if str(s.dtype).startswith('string'):
        return True
    sample = s.dropna().head(50)
    if len(sample) == 0: return False
    try:
        pd.to_numeric(sample); return False
    except (ValueError, TypeError):
        return True

obj_cols = [c for c in X.columns if is_cat_col(X[c])]
num_cols = [c for c in X.columns if c not in obj_cols]
print(f"  Categorical: {len(obj_cols)}, Numerical: {len(num_cols)}", flush=True)

cat_dim_list = []
for c in obj_cols:
    arr_X = X[c].values
    arr_test = X_test[c].values
    def to_safe_str(arr):
        out = np.empty(len(arr), dtype=object)
        for i, v in enumerate(arr):
            if v is None or (isinstance(v, float) and np.isnan(v)) or (isinstance(v, str) and v == 'nan'):
                out[i] = '__NA__'
            else:
                out[i] = str(v)
        return out
    s_X = to_safe_str(arr_X)
    s_test = to_safe_str(arr_test)
    all_vals = np.concatenate([s_X, s_test])
    uniq = sorted(set(all_vals.tolist()))
    mapping = {v: i for i, v in enumerate(uniq)}
    X[c] = np.array([mapping[v] for v in s_X], dtype=np.int32)
    X_test[c] = np.array([mapping[v] for v in s_test], dtype=np.int32)
    cat_dim_list.append(len(uniq))

for c in num_cols:
    X[c] = pd.to_numeric(X[c], errors='coerce').fillna(0).astype("float32")
    X_test[c] = pd.to_numeric(X_test[c], errors='coerce').fillna(0).astype("float32")

X_num_all = X[num_cols].values.astype("float32")
X_cat_all = X[obj_cols].values.astype("int64")
X_test_num = X_test[num_cols].values.astype("float32")
X_test_cat = X_test[obj_cols].values.astype("int64")
print(f"  X_num={X_num_all.shape}, X_cat={X_cat_all.shape}", flush=True)

# ─── FT-Transformer ────────────────────────────────────
class FTTransformer(nn.Module):
    """Feature-Tokenizer Transformer (Yandex 2021)."""
    def __init__(self, num_n, cat_dims, d_model=64, n_heads=4, n_layers=3, dropout=0.2):
        super().__init__()
        self.num_n = num_n
        self.n_cat = len(cat_dims)
        # Numerical tokenization: each numeric → token
        self.num_weight = nn.Parameter(torch.randn(num_n, d_model) * 0.02)
        self.num_bias = nn.Parameter(torch.zeros(num_n, d_model))
        # Categorical embeddings
        self.cat_embs = nn.ModuleList([nn.Embedding(d, d_model) for d in cat_dims])
        # CLS token
        self.cls_tok = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        # Transformer encoder
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 2,
            dropout=dropout, batch_first=True, activation='gelu', norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, 1)
        )

    def forward(self, x_num, x_cat):
        B = x_num.shape[0]
        # Numerical tokens: (B, num_n, d)
        num_tok = x_num.unsqueeze(-1) * self.num_weight.unsqueeze(0) + self.num_bias.unsqueeze(0)
        # Categorical tokens: (B, n_cat, d)
        if self.n_cat > 0:
            cat_toks = torch.stack(
                [emb(x_cat[:, i]) for i, emb in enumerate(self.cat_embs)],
                dim=1,
            )
            tokens = torch.cat([num_tok, cat_toks], dim=1)
        else:
            tokens = num_tok
        # Add CLS
        cls = self.cls_tok.expand(B, -1, -1)
        x = torch.cat([cls, tokens], dim=1)
        # Transformer
        x = self.transformer(x)
        # CLS head
        return self.head(x[:, 0]).squeeze(-1)

# ─── Training ──────────────────────────────────────────
N_FOLDS = 5
EPOCHS = 20
BATCH = 1024  # Smaller batch for transformer (memory)
LR = 5e-4
WD = 1e-5
PATIENCE = 4

skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=42)
oof = np.zeros(len(X))
test_pred = np.zeros(len(X_test))

print(f"\n[2/4] FT-Transformer training (epochs={EPOCHS}, batch={BATCH})", flush=True)
t0 = time.time()
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_num_all, y)):
    ts = time.time()

    scaler = StandardScaler()
    Xtr_num = scaler.fit_transform(X_num_all[tr_idx])
    Xva_num = scaler.transform(X_num_all[va_idx])
    Xte_num = scaler.transform(X_test_num)

    Xtr_num_t = torch.from_numpy(Xtr_num.astype("float32"))
    Xva_num_t = torch.from_numpy(Xva_num.astype("float32"))
    Xte_num_t = torch.from_numpy(Xte_num.astype("float32"))
    Xtr_cat_t = torch.from_numpy(X_cat_all[tr_idx])
    Xva_cat_t = torch.from_numpy(X_cat_all[va_idx])
    Xte_cat_t = torch.from_numpy(X_test_cat)
    ytr_t = torch.from_numpy(y[tr_idx].astype("float32"))

    train_ds = TensorDataset(Xtr_num_t, Xtr_cat_t, ytr_t)
    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=0)

    model = FTTransformer(num_n=Xtr_num.shape[1], cat_dims=cat_dim_list).to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=EPOCHS)
    bce = nn.BCEWithLogitsLoss()

    best_auc, best_va, best_te, no_improve = 0, None, None, 0
    for ep in range(EPOCHS):
        model.train()
        for xn, xc, yb in train_loader:
            xn, xc, yb = xn.to(device), xc.to(device), yb.to(device)
            optim.zero_grad()
            loss = bce(model(xn, xc), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()
        sched.step()

        model.eval()
        bs = 2048
        va_preds = []
        with torch.no_grad():
            for i in range(0, len(Xva_num_t), bs):
                logit = model(Xva_num_t[i:i+bs].to(device), Xva_cat_t[i:i+bs].to(device)).cpu().numpy()
                va_preds.append(logit)
        va_logit = np.concatenate(va_preds)
        va_pred = 1 / (1 + np.exp(-va_logit))
        auc = roc_auc_score(y[va_idx], va_pred)
        if auc > best_auc:
            best_auc = auc
            best_va = va_pred
            te_preds = []
            with torch.no_grad():
                for i in range(0, len(Xte_num_t), bs):
                    logit = model(Xte_num_t[i:i+bs].to(device), Xte_cat_t[i:i+bs].to(device)).cpu().numpy()
                    te_preds.append(logit)
            te_logit = np.concatenate(te_preds)
            best_te = 1 / (1 + np.exp(-te_logit))
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                break

    oof[va_idx] = best_va
    test_pred += best_te / N_FOLDS
    print(f"  Fold {fold+1}: AUC={best_auc:.5f}  best_ep={ep-no_improve+1}/{ep+1}  ({time.time()-ts:.0f}s)", flush=True)

print(f"\n[3/4] OOF AUC: {roc_auc_score(y, oof):.5f}", flush=True)
np.savez_compressed(MODELS / "oof_ft.npz", ft=oof, y=y)
np.savez_compressed(MODELS / "test_ft.npz", ft=test_pred)
print(f"\n[4/4] Done. Total: {(time.time()-t0)/60:.1f} min")
print(f"  Saved: models/oof_ft.npz, models/test_ft.npz")
